# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q statsmodels
    print('Packages installed.')


# Linear Regression Analysis In-Depth

This notebook covers linear regression with a focus on statistical inference and assumption checking, which is crucial for social science research. We will use `statsmodels` for detailed output.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import scipy.stats as stats

sns.set_style('whitegrid')

## 1. Generate Synthetic Data
We will simulate a dataset representing the relationship between 'Education Level' (years), 'Experience' (years), and 'Income' ($000s). We add some random noise to mimic real-world data.

In [ ]:
np.random.seed(42)
n_samples = 200

# Predictors
education = np.random.normal(14, 2, n_samples)  # Mean 14 years, SD 2
experience = np.random.normal(10, 5, n_samples) # Mean 10 years, SD 5

# Dependent Variable (Income)
# True relationship: Income = 20 + 3*Education + 1.5*Experience + Noise
noise = np.random.normal(0, 5, n_samples)
income = 20 + 3 * education + 1.5 * experience + noise

df = pd.DataFrame({
    'Education': education,
    'Experience': experience,
    'Income': income
})

print(df.head())

## 2. Exploratory Data Analysis (EDA)
Check for linearity using scatter plots.

In [ ]:
sns.pairplot(df)
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

## 3. Fit the OLS Model
We use Ordinary Least Squares (OLS) from `statsmodels`. We must manually add a constant (intercept).

In [ ]:
X = df[['Education', 'Experience']]
y = df['Income']

# Add constant for intercept
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

## 4. Assumption Checking
Linear regression relies on several assumptions. Violating these can lead to biased estimates.

### A. Normality of Residuals
Residuals should be normally distributed. We check this with a Q-Q plot and a histogram.

In [ ]:
residuals = model.resid

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Histogram
sns.histplot(residuals, kde=True, ax=ax[0])
ax[0].set_title('Histogram of Residuals')

# Q-Q Plot
sm.qqplot(residuals, line='45', fit=True, ax=ax[1])
ax[1].set_title('Q-Q Plot')

plt.show()

### B. Homoscedasticity
The variance of residuals should be constant across all levels of the independent variables. We check this by plotting Residuals vs. Fitted Values. A cone shape indicates heteroscedasticity.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(model.fittedvalues, residuals)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values')
plt.show()

### C. Multicollinearity
Predictors should not be too highly correlated. We check the Variance Inflation Factor (VIF). A VIF > 5 or 10 indicates potential trouble.

In [ ]:
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

print(vif_data)

## Conclusion
This notebook demonstrated a full regression workflow. We generated data, fit a model, and crucially, validated the model assumptions. In a real social science context, interpreting the coefficients (Effect size) and p-values (Significance) alongside these diagnostic checks is essential for robust conclusions.